In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

if module_path not in sys.path:
    sys.path.append(module_path)

In [13]:
# Set up Dask cluster using Docker Python SDK
import docker

# Initialize Docker client
docker_client = docker.from_env()

# Create a Docker network
network_name = "dask-network"

try:
    docker_client.networks.get(network_name)
except docker.errors.NotFound:
    docker_client.networks.create(network_name, driver="bridge")

# Define paths for volume mounting
host_file_path = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'  # Host machine path
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

host_nc_path = r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\experiment_code\saved_on_disk.nc'
container_nc_path = '/data/saved_on_disk.nc'

# Volume mapping
volumes = {
    host_file_path: {'bind': container_file_path, 'mode': 'ro'},  # 'ro' means read-only
    host_nc_path: {'bind': container_nc_path, 'mode': 'ro'},
    'timetracker.py': {'bind': '/app/timetracker.py', 'mode': 'ro'},  # Mount the timetracker script
}
'''
# Run Dask Scheduler
scheduler = docker_client.containers.run(
    "adrianomdocker/rrtest",
    command="dask-scheduler",
    name="dask-scheduler",
    network=network_name,
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787},
)'''

'''
scheduler = docker_client.containers.run(
    "adrianomdocker/rrtest",
    command="dask-scheduler --preload dask_memusage --memusage-csv /tmp/memusage.csv",
    name="dask-scheduler",
    network=network_name,
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787},
)
'''
scheduler = docker_client.containers.run(
    "adrianomdocker/rrtest",
    command="dask-scheduler --preload dask_memusage --preload /app/timetracker.py --timetracker-csv /tmp/timetracker.csv --memusage-csv /tmp/memusage.csv",
    name="dask-scheduler",
    network=network_name,
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787}
)

print(f"Dask Scheduler started with ID {scheduler.id}")

# Run Dask Workers
num_workers = 2  # Specify the number of workers you want
n_threads = 1
container_mem_limit = "8GB"
dask_mem_limit = "8GB"

worker_ids = []
for i in range(num_workers):
    worker = docker_client.containers.run(
        "adrianomdocker/rrtest",
        command=f"dask-worker dask-scheduler:8786 --nthreads {n_threads} --memory-limit {dask_mem_limit}",
        name=f"dask-worker-{i+1}",
        network=network_name,
        detach=True,
        mem_limit=container_mem_limit,
        volumes=volumes
)
    worker_ids.append(worker.id)
    print(f"Dask Worker {i+1} started with ID {worker.id}")

# Connect to the Dask Scheduler
#dask_client = Client('tcp://localhost:8786')

print("Connected to Dask Scheduler")

# Monitor the Dask Dashboard
print("Dask dashboard available at http://localhost:8787")

# The cluster is now set up and can be used within the script or interactively.

Dask Scheduler started with ID 601f64a89a8fc94f0ab2f32830c76bac4cda5c2cd3ad9c26dc8db42ad7f4d219
Dask Worker 1 started with ID f5a2f06dc5d7240af4dfc5ee9b4d36a892f554b4cb0e0f831acebfe853ceef1f
Dask Worker 2 started with ID ed080df6be21bf38c9c7670f041afd3f7b780a39de76616fa77769500470ac54
Connected to Dask Scheduler
Dask dashboard available at http://localhost:8787


In [5]:
from dask.distributed import Client
dask_client = Client('tcp://localhost:8786')

2024-09-13 13:46:47,552 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [11]:
from dask_plugins import EEPlugin

ee_plugin = EEPlugin(container_file_path)
dask_client.register_plugin(ee_plugin)

{'tcp://172.20.0.3:33869': {'status': 'OK'},
 'tcp://172.20.0.4:36193': {'status': 'OK'}}

In [13]:
# EE credentials with a JSON key
import json
import ee
json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

In [ ]:
###############################
### TESTING MY PACKAGE CODE ###
###############################

In [14]:
# Earth Engine xarray
import sys
import os
import ee
import os

#ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project='robust-raster')
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
}

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }

landsat_xarray = EarthEngineData(parameters)

In [15]:
ds = landsat_xarray.dataset

In [1]:
# Template xarray based on Earth Engine
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


In [10]:
# Load the data using Dask Delayed!
import xarray as xr
import dask

chunk_size  = {
            'time': 3,
            'X': 512,
            'Y': 256
}
# Define a delayed function that opens the dataset
@dask.delayed
def load_dataset_on_worker():
    return xr.open_dataset('/data/saved_on_disk.nc', chunks=chunk_size)

# Trigger the delayed execution, this will happen on the worker
delayed_ds = load_dataset_on_worker()

# Optionally, you can convert the delayed object into a Dask-backed xarray Dataset
ds = delayed_ds.compute()

# Now ds can be used for further computation on the Dask workers

In [ ]:
import xarray as xr
import dask.array as da
import dask
import psutil
import time

def log_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024**2  # Memory usage in MB


def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

def user_function_wrapper(ds, user_func, *args, **kwargs):
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    return ds_output


def tune_user_function(chunk, user_func, user_func2, *args, **kwargs):

    result_chunk = user_func2(ds, user_func, *args, kwargs)
    start_time = time.time()
    result_chunk.compute()
    end_time = time.time()
        
    wall_time = end_time - start_time
    memory_usage = log_memory_usage()
    print(f"Wall time: {wall_time:.2f} seconds")
    print(f"Memory usage: {memory_usage:.2f} MB")

    # Apply the processing function to this chunk
    #processed_chunk = user_func(one_chunk)

    return result_chunk

def test_run(ds, user_func, user_func2, *args, **kwargs):
    # Dynamically determine dimension names
    dim_names = list(ds.dims.keys())
            
    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)

    test = xr.map_blocks(tune_user_function,
                            one_chunk, 
                            args=(user_func, user_func2) + args, 
                            kwargs=kwargs)

result = test_run(ds, process_chunk_as_pandas, user_function_wrapper)

In [11]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)
result.compute()

C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src\udf_tuner.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dim_names = list(ds.dims.keys())


Wall time: 1.30 seconds
Memory usage: 165.42 MB


<xarray.Dataset> Size: 5MB
Dimensions:  (X: 512, Y: 256, time: 3)
Coordinates:
  * X        (X) float64 4kB -4.216e+05 -4.215e+05 ... -3.706e+05 -3.705e+05
  * Y        (Y) float64 2kB -5.992e+05 -5.991e+05 ... -5.738e+05 -5.737e+05
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
Data variables:
    SR_B4    (time, X, Y) float32 2MB 0.3455 0.1448 0.7745 ... 0.9714 0.3537
    ndvi     (time, X, Y) float32 2MB 0.691 0.2896 1.549 ... 0.5705 1.943 0.7074
    SR_B5    (time, X, Y) float32 2MB 0.3455 0.1448 0.7745 ... 0.9714 0.3537

In [7]:
import xarray as xr
import dask.array as da
import numpy as np
import dask
import psutil
import time

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    return df

def user_function_wrapper(ds, user_func, *args, **kwargs):
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    return ds_output

def log_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024**2  # Memory usage in MB

best_composite_score = 999.

def find_optimal_chunk_size(ds, best_composite_score, chunk_size):
    '''
    Recursively find the optimal chunk size for the user function.
    '''
    wall_time_normalized = -1.
    mem_usage_normalized = -1.
    elements_per_chunk = -1

    # Run a chunk of my data in user_function_wrapper
    # Dynamically determine dimension names
    dim_names = list(ds.sizes.keys())

    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)

    # Extract the dimensions of the single chunk
    chunk_shape = one_chunk.sizes
    elements_per_chunk = np.prod(list(chunk_shape.values()))
    print("elements_per_chunk: ", elements_per_chunk)
    # Monitor the wall time and memory usage
    start_time = time.time()
    result = user_function_wrapper(one_chunk, process_chunk_as_pandas)
    result.compute()
    end_time = time.time()
    wall_time = end_time - start_time
    memory_usage = log_memory_usage()

    # Normalize the wall time and memory usage
    mem_usage_normalized = memory_usage / elements_per_chunk
    wall_time_normalized = wall_time / elements_per_chunk

    composite_score = wall_time_normalized + mem_usage_normalized

    # A best composite score would be the SMALLEST score!
    if composite_score < best_composite_score:
        print("Composite score: ", composite_score)
        print("Best composite score: ", best_composite_score)
        print("Memory usage: ", memory_usage)
        print("Wall time: ", wall_time)
        best_composite_score = composite_score
        new_chunk_size = {key: value * 2 for key, value in chunk_size.items()}
        print(new_chunk_size)
        ds = ds.chunk(new_chunk_size)
        find_optimal_chunk_size(ds, best_composite_score, new_chunk_size)
    else:
        return result, best_composite_score

# When I rechunk, it goes back to open_dataset to do so...
# It would be nice if it didn't do that...
# Just rechunk the data and not open...

In [ ]:
# CHATGPT optimized function optimizer
import dask
from dask.distributed import get_client
import xarray as xr
import dask.array as da
import numpy as np
import dask
import psutil
import time
import dask
from dask.distributed import get_client

def get_worker_memory_usage(client):
    # Get the scheduler info, which includes all worker information
    workers = client.scheduler_info()['workers'].values()

    # Extract the memory usage from the 'metrics' section for each worker
    total_memory = sum(worker['metrics']['memory'] for worker in workers) / 1024**2  # Convert to MB
    return total_memory


def find_optimal_chunk_size(ds, best_composite_score, chunk_size, max_iterations=10, iteration=0):
    '''
    Recursively find the optimal chunk size for the user function.
    '''
    if iteration >= max_iterations:
        print("Max iterations reached")
        return ds, best_composite_score

    wall_time_normalized = -1.
    mem_usage_normalized = -1.
    elements_per_chunk = -1

    # Dynamically determine dimension names
    dim_names = list(ds.sizes.keys())

    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)

    # Extract the dimensions of the single chunk
    chunk_shape = one_chunk.sizes
    elements_per_chunk = np.prod(list(chunk_shape.values()))

    # Monitor the wall time and memory usage
    start_time = time.time()
    result = user_function_wrapper(one_chunk, process_chunk_as_pandas)
    result.persist()  # Keep the computation in Dask memory, without computing fully
    end_time = time.time()
    wall_time = end_time - start_time

    # Gather memory usage across all workers
    client = get_client()
    memory_usage = get_worker_memory_usage(client)

    # Normalize the wall time and memory usage
    mem_usage_normalized = memory_usage / elements_per_chunk
    wall_time_normalized = wall_time / elements_per_chunk

    composite_score = wall_time_normalized + mem_usage_normalized

    # A best composite score would be the SMALLEST score!
    if composite_score < best_composite_score:
        print(f"New Best Composite score: {composite_score} (was {best_composite_score})")
        print(f"Memory usage: {memory_usage} MB, Wall time: {wall_time} seconds")
        best_composite_score = composite_score
        new_chunk_size = {key: value * 2 for key, value in chunk_size.items()}
        print(f"Rechunking to: {new_chunk_size}")
        ds = ds.chunk(new_chunk_size)
        return find_optimal_chunk_size(ds, best_composite_score, new_chunk_size, max_iterations, iteration + 1)
    else:
        print("Stopping as no better score found.")
        return result, best_composite_score

In [ ]:
result, best_composite_score = find_optimal_chunk_size(ds, best_composite_score, chunk_size)